In [ ]:

import numpy as np
import pycuda.autoinit
import pycuda.driver as cuda
from pycuda.compiler import SourceModule
import time

def validate_matrix_dims(mat1, mat2):
    """Проверка, что умножение матриц возможно."""
    if mat1.shape[1] != mat2.shape[0]:
        raise ValueError("Размеры матриц не соответствуют правилам умножения")

def cpu_matrix_mult(mat1, mat2):
    """Умножение матриц на CPU."""
    validate_matrix_dims(mat1, mat2)

    output = np.zeros((mat1.shape[0], mat2.shape[1]))

    start = time.time()
    for row in range(mat1.shape[0]):
        for col in range(mat2.shape[1]):
            for index in range(mat1.shape[1]):
                output[row, col] += mat1[row, index] * mat2[index, col]
    end = time.time()

    return output, end - start

def gpu_matrix_mult(mat1, mat2):
    """Умножение матриц на GPU с использованием PyCUDA."""
    validate_matrix_dims(mat1, mat2)

    # CUDA Kernel
    mod = SourceModule("""
    __global__ void matrixMulKernel(float *A, float *B, float *C, int N) {
        int row = blockIdx.y * blockDim.y + threadIdx.y;
        int col = blockIdx.x * blockDim.x + threadIdx.x;

        if(row < N && col < N) {
            float sum = 0.0;
            for (int k = 0; k < N; k++) {
                sum += A[row * N + k] * B[k * N + col];
            }
            C[row * N + col] = sum;
        }
    }
    """)

    # Размер матриц
    N = mat1.shape[0]

    # Аллокация памяти на GPU
    mat1_gpu = cuda.mem_alloc(mat1.nbytes)
    mat2_gpu = cuda.mem_alloc(mat2.nbytes)
    result_gpu = cuda.mem_alloc(mat1.nbytes)  # результат такой же размерности, что и мат1 и мат2

    # Копирование данных из CPU в GPU
    cuda.memcpy_htod(mat1_gpu, mat1)
    cuda.memcpy_htod(mat2_gpu, mat2)

    # Вызываем CUDA Kernel
    matrix_mul = mod.get_function("matrixMulKernel")

    # Настройка сетки и блоков
    block_size = (16, 16, 1)
    grid_size = (int(np.ceil(N / 16)), int(np.ceil(N / 16)), 1)

    start = time.time()
    matrix_mul(mat1_gpu, mat2_gpu, result_gpu, np.int32(N), block=block_size, grid=grid_size)
    cuda.Context.synchronize()
    end = time.time()

    # Копирование результата с GPU в CPU
    result = np.empty_like(mat1)
    cuda.memcpy_dtoh(result, result_gpu)

    return result, end - start

if __name__ == "__main__":
    shape = (50, 50)
    mat1 = np.random.random(shape).astype(np.float32)
    mat2 = np.random.random(shape).astype(np.float32)

    cpu_result, cpu_exec_time = cpu_matrix_mult(mat1, mat2)
    gpu_result, gpu_exec_time = gpu_matrix_mult(mat1, mat2)

    # print(f"Результат на CPU:\n{cpu_result}")
    print(f"Время выполнения на CPU: {cpu_exec_time:.6f} сек")

    # print(f"Результат на GPU:\n{gpu_result}")
    print(f"Время выполнения на GPU: {gpu_exec_time:.6f} сек")


Время выполнения на CPU: 0.095404 сек
Время выполнения на GPU: 0.000087 сек


In [ ]:
!pip install pycuda

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 67.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.8/98.8 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 11.2 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2026.1-cp312-cp312-linux_x86_64.whl size=659447 sha256=1e0a6ac7af28595b7ada0ceb969ccfb1cfc6b95dd593d602ce2684cdf4f14e26
  Stored in directory: /root/.cache/pip/wheels/90/2a/71/75ec0cc316cc0ff494bfffa2935e02580129cb7f859a0cfd8f
Successfully built pycuda


In [ ]:
sizes = [50, 100, 200, 400, 800, 1600, 3200, 6400, 12800]

for n in sizes:
    shape = (n, n)
    mat1 = np.random.random(shape).astype(np.float32)
    mat2 = np.random.random(shape).astype(np.float32)

    print(f"\n>>> Тестируем размер: {n}x{n}")

    # CPU Test
    _, t_cpu = cpu_matrix_mult(mat1, mat2)
    print(f"CPU: {t_cpu:.4f} сек")

    # GPU Test
    _, t_gpu = gpu_matrix_mult(mat1, mat2)
    print(f"GPU: {t_gpu:.4f} сек")

    speedup = t_cpu / t_gpu
    print(f"Ускорение: {speedup:.2f}x")


>>> Тестируем размер: 50x50
CPU: 0.0892 сек
GPU: 0.0001 сек
Ускорение: 1000.62x

>>> Тестируем размер: 100x100
CPU: 0.7128 сек
GPU: 0.0001 сек
Ускорение: 8818.80x

>>> Тестируем размер: 200x200
CPU: 6.8386 сек
GPU: 0.0002 сек
Ускорение: 44401.20x

>>> Тестируем размер: 400x400
CPU: 50.4062 сек
GPU: 0.0005 сек
Ускорение: 94467.80x

>>> Тестируем размер: 800x800
CPU: 403.2449 сек
GPU: 0.0045 сек
Ускорение: 90339.26x

>>> Тестируем размер: 1600x1600
CPU: 3156.5698 сек
GPU: 0.0363 сек
Ускорение: 87011.13x

>>> Тестируем размер: 3200x3200
